# Геолокация по двум эмбеддингам · подбор по MAE

По умолчанию ноутбук показывает опубликованные измерения. Для повторения обучения выберите `reproduce` в следующей ячейке и включите GPU в настройках Colab.

Обе модели получают 512 признаков GeoCLIP и 768 признаков StreetCLIP. Ошибка снимка — расстояние WGS84 в километрах между предсказанной и известной точкой. **MAE** усредняет эти расстояния по снимкам. **RMSE** возводит их в квадрат перед усреднением, поэтому крупные промахи сильнее влияют на выбор модели. Для каждого критерия сохраняется свой победитель пятифолдовой проверки.

[Ноутбук подбора по RMSE](train250k_final500.ipynb) · [Опубликованный результат RMSE](artifacts/full500/results.json) · [Опубликованный результат MAE](artifacts/mae_cv/results.json)

[Как обе модели превращают признаки снимка в координаты](docs/models-overview.svg)

![Обучение и оценка на отложенных снимках](docs/workflow-mae.svg)

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess
import sys
from IPython.display import Markdown, display

MODE = "published"  # @param ["published", "reproduce"]
REPO_DIR = "/content/geovec"  # @param {type:"string"}
TRAIN_250K_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
GEO_CV_ROOT = "/content/train250k-cv"  # @param {type:"string"}
MOUNT_DRIVE = True  # @param {type:"boolean"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

if MODE not in {"published", "reproduce"}:
    raise ValueError("Выберите published или reproduce.")
if MODE == "reproduce" and MOUNT_DRIVE and not Path("/content/drive/MyDrive").is_dir():
    try:
        from google.colab import drive
    except ImportError:
        pass
    else:
        drive.mount("/content/drive")

checkout = Path(REPO_DIR).expanduser().resolve()
repo = next((p for p in (checkout, Path.cwd(), Path.cwd().parent)
             if (p / "train250k_mae.ipynb").is_file()), None)
if repo is None:
    checkout.parent.mkdir(parents=True, exist_ok=True)
    if checkout.exists() and any(checkout.iterdir()):
        raise FileExistsError(f"В {checkout} нет train250k_mae.ipynb. Укажите актуальную копию репозитория.")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/DenisChistov/geovec.git",
                    str(checkout)], check=True)
    repo = checkout
src = repo / "src"
work = Path(GEO_CV_ROOT).expanduser().resolve()
data = Path(TRAIN_250K_DIR).expanduser().resolve()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

env = {**os.environ, "TRAIN_250K_DIR": str(data), "GEO_CV_ROOT": str(work)}
env["PYTHONPATH"] = os.pathsep.join([str(src / "screen"), str(src), env.get("PYTHONPATH", "")])

def run_src(script, *args):
    subprocess.run([sys.executable, "-u", str(src / script), *map(str, args)],
                   cwd=repo, env=env, check=True)

## Общие прогнозы для обеих целей

Для воспроизведения укажите `GEO_CV_ROOT` от завершённого RMSE-ноутбука. Нужны рабочие файлы: подготовленные массивы, внешние и внутренние прогнозы V3, прогнозы специалистов и обученные веса `full500`. Публичная папка `artifacts/` содержит численные результаты. С пустым рабочим каталогом следующая ячейка выполнит этапы общего эксперимента и сохранит их для повторного использования. Исходный `train_manifest.parquet` и шарды эмбеддингов должны находиться в `TRAIN_250K_DIR`.

Фиксированное разбиение содержит 212 500 снимков разработки и 37 500 тестовых. Пять внешних фолдов проверяют модели на 42 500 снимках каждый. Внефолдовые прогнозы для обучения ансамблей получены моделями, которые обучались на других строках.

In [ ]:
if MODE == "reproduce":
    if INSTALL_DEPENDENCIES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(repo / "requirements.txt")], check=True)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Для воспроизведения выберите GPU в настройках среды Colab.")
    print("GPU", torch.cuda.get_device_name())
    shared = work / "full500"
    shared_cv = shared / "cv-summary.json"
    if not (work / "split.json").is_file():
        run_src("prepare.py")
    split = read_json(work / "split.json")
    assert (split["trainval"], split["test"], split["fold_sizes"]) == (212500, 37500, [42500]*5)
    v3 = work / "v3"
    bundles = [v3 / f"bundle-o{i}.npz" for i in range(5)]
    bundles += [v3 / f"bundle-o{i}-i{j}.npz" for i in range(5) for j in range(3)]
    if not (v3 / "cv-summary.json").is_file() or not all(p.is_file() for p in bundles):
        run_src("run_v3.py", "cv")
    if not (work / "v4/cv-summary.json").is_file():
        run_src("run_v4.py", "cv")
    if not shared_cv.is_file():
        if not all((work / "v5" / f"literature-predictions-{i}.npz").is_file() for i in range(5)):
            run_src("run_research.py")
        if not (work / "screen500/summary.json").is_file():
            run_src("screen/screen.py")
        run_src("screen/full_search.py")
    if "strong_fusion" not in read_json(shared_cv):
        run_src("screen/strong_fusion.py")
    if "diversity_fusion" not in read_json(shared_cv):
        if not (work / "diversity500/selection.json").is_file():
            run_src("screen/diversity_select.py")
        run_src("screen/diversity_cv.py")
    if "tuned_fusion" not in read_json(shared_cv):
        run_src("screen/tune_fusion.py")
    if not (shared / "results.json").is_file():
        run_src("screen/final_full.py")
    rmse_hash = file_hash(shared_cv)
    print("Общий эксперимент", shared, "· фолды V3", len(bundles))

## Подбор отдельной модели по MAE

Сравниваются решения по апостериорным распределениям ячеек, включая географическую медиану, локальную моду и смешивание с прогнозами поиска. Также проверяются устойчивые комбинации общих экспертов и модели, определяющие их веса для каждого изображения. Победитель имеет минимальный средний WGS84 MAE по всем 212 500 внефолдовым прогнозам. При равенстве используется RMSE, затем имя конфигурации. Пятифолдовая оценка служит для подбора модели.

In [ ]:
if MODE == "reproduce":
    run_src("screen/mae_search.py")
    assert file_hash(shared_cv) == rmse_hash
    results_root = work
else:
    results_root = repo / "artifacts"

mae_cv = read_json(results_root / "mae_cv/cv-summary.json")
ranking = sorted(mae_cv["candidates"], key=lambda x: (x["cv_mae_km"], x["cv_rmse_km"], x["name"]))
assert mae_cv["winner"] == ranking[0]["name"]
print("Конфигураций", len(ranking), "· внешних фолдов", len(ranking[0]["fold_mae_km"]))
lines = ["| Конфигурация | CV MAE, км | CV RMSE, км |", "|---|---:|---:|"]
for row in ranking[:5]:
    lines.append(f"| `{row['name']}` | {row['cv_mae_km']:.2f} | {row['cv_rmse_km']:.2f} |")
display(Markdown("\n".join(lines)))

## Результаты на одном тестовом наборе

В режиме воспроизведения финальный этап фиксирует выбранное правило, при необходимости обучает его на всех внефолдовых прогнозах и сохраняет 37 500 тестовых предсказаний до чтения координат для оценки. Итоги MAE записываются в `mae_cv/`, итоги RMSE хранятся в `full500/`. Этот тест использовался в предыдущих версиях исследования. Для независимой проверки нужен новый закрытый набор.

In [ ]:
if MODE == "reproduce":
    run_src("screen/mae_final.py")
    assert file_hash(shared_cv) == rmse_hash
rmse_result = read_json(results_root / "full500/results.json")
mae_result = read_json(results_root / "mae_cv/results.json")
assert rmse_result["split_sha256"] == mae_result["split_sha256"] == mae_cv["split_sha256"]
assert mae_result["rmse_cv_sha256"] == rmse_result["cv_sha256"]
assert mae_result["winner"] == mae_cv["winner"]
rows = [("RMSE", rmse_result), ("MAE", mae_result)]
table = ["| Цель подбора | CV MAE, км | CV RMSE, км | Тест MAE, км | Тест RMSE, км |",
         "|---|---:|---:|---:|---:|"]
for objective, result in rows:
    cv, test = result["cv"], result["test"]
    assert test["n"] == 37500
    table.append(f"| {objective} | {cv['cv_mae_km']:.2f} | {cv['cv_rmse_km']:.2f} | "
                 f"{test['mae_km']:.2f} | {test['rmse_km']:.2f} |")
display(Markdown("\n".join(table)))
for objective, result in rows:
    print(f"Победитель {objective}: {result['winner']}")
delta_mae = mae_result["test"]["mae_km"] - rmse_result["test"]["mae_km"]
delta_rmse = mae_result["test"]["rmse_km"] - rmse_result["test"]["rmse_km"]
print(f"Изменение тестовых метрик при подборе по MAE: MAE {delta_mae:+.2f} км, RMSE {delta_rmse:+.2f} км")

## Прогноз новых координат

После обучения обе команды принимают массив NumPy размера `N × 1280`. Сначала идут 512 координат GeoCLIP, затем 768 координат StreetCLIP. Файл CSV содержит широту и долготу WGS84 в градусах. Выполните команду из корня проекта, указав свой рабочий каталог и файл эмбеддингов.

```bash
PYTHONPATH=src/screen:src python src/screen/predict_mae.py --reference /content/train250k-cv --input new_embeddings.npy --output mae_coordinates.csv
PYTHONPATH=src/screen:src python src/screen/predict_final.py --reference /content/train250k-cv --input new_embeddings.npy --output rmse_coordinates.csv
```

[Прогноз модели MAE](src/screen/predict_mae.py) · [Прогноз модели RMSE](src/screen/predict_final.py)